<a href="https://colab.research.google.com/github/csu-techhub/QOS-Labs/blob/main/Lab6_extension_H2O.ipynb" target="_parent">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Assignment 2 (Lab 6 extension) — The Shape of Water: Two Geometry Parameters
**Quantum Optimization and Simulation — VQE Laboratory Series**

**Runtime:** about 10 minutes in Colab for the whole notebook (the optional IBM hardware part is skipped unless you turn it on).

In Lab 6 we drew the potential-energy curve of H₂, whose shape depends on one number, the bond length $R$. At the end of the lecture (**Module 4, Lesson 5**, slide "Overall VQE Process: H2") we asked: *Is $R$ the only parameter for geometry optimization?* The next slides answered with **water**: for H₂O the outer loop adjusts the O–H distance **and** the H–O–H angle. This assignment carries out that example.

> **How to read this notebook.** Every code cell starts with a label such as `# [Cell 5]`, so the review questions at the end can refer to it. Boxes marked **📘 Lecture link** point to the Module 3 / Module 4 slides and the earlier labs that each step builds on.

## Learning objectives
- Describe water's **two geometry parameters** (bond length $R$ and bond angle $\theta$) and how they enter the outer loop.
- Use the **cars and lanes** picture to describe water's full (unreduced) problem: 10 electrons in 14 lanes, and the lane changes in its ansatz.
- Explain, in plain words, the three steps that shrink 14 qubits to 5.
- Run VQE over both geometry parameters (inner loop + outer loop) and find the lowest-energy shape.
- See what a noisy (FakeSherbrooke) device does to the result.
- *(Optional)* Measure the water energy on a real IBM quantum computer.

## Where each idea comes from
| In this notebook | Lecture | Earlier lab |
|---|---|---|
| Two geometry parameters, $R$ and the angle | Module 4, Lesson 5: "Overall VQE Process: H2O" | Lab 6 (one parameter, $R$) |
| Electrons = cars, spin orbitals = lanes | Module 3, Lesson 3: "Preparing a Trial State for H2 – Configuration Mix" | Lab 4, Part C1 |
| Hartree–Fock reference, lane changes, UCCSD | Module 3, Lesson 4: CC → UCC → UCCSD | Lab 4, Part C2; Lab 5, Optional |
| Shrinking the problem (redundant qubits, tapering) | Module 3, Lesson 4: "Reduced Qubit Mapping" | Lab 4 (4 → 2 qubits) |
| Pauli Hamiltonian and mappings (Jordan–Wigner, parity) | Module 4, Lesson 2: "Measuring Total Energy of H2" | Lab 5, Optional Step 2 |
| Measuring $\langle H\rangle$ in the Z, X and Y bases | Module 4, Lessons 3–4 | Lab 5, Part A |
| COBYLA adjusts the knobs | Module 4, Lesson 4: "Adjust Parameters (θ's)" | Lab 5, Part C |
| Inner loop vs. outer loop | Module 4, Lesson 5: "Overall VQE Process" | Lab 6, Part A |
| Noise on today's hardware (NISQ) | Module 3, Lesson 1: "Hybrid Algorithms for Today's Hardware (NISQ)" | Lab 6, Part B |
| VQE is expensive | Module 4, Lesson 5: "VQE is Expensive" | Lab 6, Reflection |

## Meet H₂O (water)
- **Three atoms:** one oxygen (O) in the middle and two hydrogens (H).
- **Ten electrons:** oxygen brings 8 and each hydrogen brings 1. In H₂ there were 2. In our picture: **10 cars instead of 2.**
- **A bent shape:** water looks like a "V", not a straight line. Oxygen keeps some of its electrons to itself (they do not take part in the O–H bonds), and those electrons push the two hydrogens toward each other. In real water the angle is about **104.5°**.

> **📘 Lecture link — Module 3, Lesson 1.** A hydrogen atom's electron is a probability cloud (the 1s orbital), and two H atoms form H₂ because they **share** their 1s electrons. Water bonds the same way: each O–H bond is a shared pair of electrons, one from O and one from H.

As in Lab 6, we compare VQE with the simple **Hartree–Fock** starting point (all cars in their lowest lanes).


In [ ]:
# [Cell 1]
# Run once in a fresh Google Colab session (pyscf is the chemistry driver).
%pip -q install "pylatexenc" "pyscf" "qiskit~=2.5" "qiskit-aer~=0.17" "qiskit-algorithms~=0.4" "qiskit-nature~=0.8" "qiskit-ibm-runtime~=0.46" "python-dateutil<3"

### Setup
The same tools as Lab 6, plus PySCF and Qiskit Nature, which build the H₂O Hamiltonian for us (as in the optional part of Lab 5).

In [ ]:
# [Cell 2]
import itertools, time, warnings
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, SparsePauliOp
from qiskit_aer import AerSimulator

from scipy.sparse import SparseEfficiencyWarning
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=SparseEfficiencyWarning)

SEED = 123
SHOTS = 4096

# A new simulator seed for every energy estimate, so each estimate sees fresh shot noise
# (still reproducible from run to run).
_seed_counter = itertools.count(SEED)

def parity_expectation(counts, qubits):
    "Expectation of Z on the listed qubits: bit 0 -> +1, bit 1 -> -1 (same as Lab 6)."
    total = sum(counts.values())
    result = 0.0
    for bits, count in counts.items():
        value = 1
        for q in qubits:
            bit = int(bits[-1 - q])
            value *= 1 if bit == 0 else -1
        result += value * count / total
    return result

def h2o_geometry(R, theta):
    "Water with both O-H bonds of length R (Angstrom) and H-O-H angle theta (degrees). O at the origin."
    half = np.deg2rad(theta) / 2
    x, z = R * np.sin(half), R * np.cos(half)
    return f"O 0 0 0; H {x:.6f} 0 {z:.6f}; H {-x:.6f} 0 {z:.6f}"

## The geometry: two parameters, $R$ and $\theta$
A molecule's shape is described by **geometric parameters**: bond lengths and bond angles. These are the knobs of the **outer loop**.

- For H₂ there was only one: the H–H distance $R$.
- For water we assume the two O–H bonds are **equal**. That leaves **two** parameters: the O–H bond length $R$ and the H–O–H angle $\theta$.

> **📘 Lecture link — Module 4, Lesson 5, "Overall VQE Process: H2O".** The slide says: *"For H₂O, two geometry parameters are: O–H distances and H–O–H angle. (It could be three with $r_1$ and $r_2$.)"* It labels them $r$ and $\alpha$ ("Adjust *r* and *α*, Hamiltonian changes"). In this notebook the bond length is $R$ and the angle is $\theta$ (in degrees), as in the code.

So the outer loop now searches a **plane** of geometries, $(R, \theta)$, instead of a line. In the code, `h2o_geometry(R, theta)` puts O at the origin and the two hydrogens at distance $R$, each tilted by $\theta/2$ from the vertical axis.

**NOTE: To keep the run time reasonable we scan one parameter at a time**: $R$ with $\theta$ fixed at 100°, and $\theta$ with $R$ fixed at 1.0 Å. The two scans cross at $(1.0\ \text{Å}, 100°)$. The right panel below shows the geometries we will scan in Part A.

In [ ]:
# [Cell 3]
from matplotlib.patches import Circle, FancyArrowPatch, Arc

def draw_atom(ax, x, y, label, radius, color):
    ax.add_patch(Circle((x, y), radius, color=color, ec="black", lw=1.2, zorder=3))
    ax.text(x, y, label, ha="center", va="center", fontsize=13, fontweight="bold", zorder=4)

def dim_arrow(ax, p, q, text, offset):
    "Double-headed dimension arrow from p to q, label shifted by offset."
    ax.add_patch(FancyArrowPatch(p, q, arrowstyle="<|-|>", mutation_scale=14, color="tab:red", lw=1.8))
    ax.text((p[0] + q[0]) / 2 + offset[0], (p[1] + q[1]) / 2 + offset[1], text,
            ha="center", va="center", fontsize=13, color="tab:red")

# Geometries scanned in Part A: one parameter at a time, crossing at (1.0 Å, 100°)
R_scan = [0.8, 0.9, 1.0, 1.1, 1.25, 1.4]
theta_scan = [80, 90, 100, 110, 125, 140]
R_FIXED, THETA_FIXED = 1.0, 100
geometries = [(R, THETA_FIXED) for R in R_scan] + [(R_FIXED, t) for t in theta_scan if t != THETA_FIXED]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1.1, 1]})

# Left: the molecule with its two parameters
theta_draw, R_draw = np.deg2rad(104.5), 1.3
h1 = (-R_draw * np.sin(theta_draw / 2), -R_draw * np.cos(theta_draw / 2))
h2 = (R_draw * np.sin(theta_draw / 2), -R_draw * np.cos(theta_draw / 2))
for h in (h1, h2):
    ax1.plot([0, h[0]], [0, h[1]], color="gray", lw=4, zorder=1)
draw_atom(ax1, 0, 0, "O", 0.34, "#f4b6b6")
draw_atom(ax1, *h1, "H", 0.22, "white")
draw_atom(ax1, *h2, "H", 0.22, "white")
for h, side in ((h1, -1), (h2, +1)):                       # dimension arrows, offset outward from the bonds
    n = np.array([-h[1], h[0]]) / np.hypot(*h) * side * 0.3
    dim_arrow(ax1, tuple(n), tuple(np.array(h) + n), "$R$", tuple(1.0 * n))
start = np.rad2deg(-np.pi / 2 - theta_draw / 2)
ax1.add_patch(Arc((0, 0), 1.0, 1.0, theta1=start, theta2=start + np.rad2deg(theta_draw), color="tab:blue", lw=2))
ax1.text(0, -0.72, r"$\theta$", ha="center", fontsize=15, color="tab:blue")
ax1.text(0, 0.55, "electrons kept by oxygen push the H atoms together", ha="center", fontsize=9, color="dimgray")
ax1.set_aspect("equal"); ax1.axis("off"); ax1.set_xlim(-1.6, 1.6); ax1.set_ylim(-1.35, 0.85)
ax1.set_title("H₂O: both O–H bonds = $R$, angle H–O–H = $\\theta$\n→ TWO geometry parameters")

# Right: the plane of geometries and the points we scan
ax2.plot(R_scan, [THETA_FIXED] * len(R_scan), "o-", color="tab:orange", label=f"scan R  (θ = {THETA_FIXED}°)")
ax2.plot([R_FIXED] * len(theta_scan), theta_scan, "s-", color="tab:purple", label=f"scan θ  (R = {R_FIXED} Å)")
ax2.plot(0.958, 104.5, "k*", ms=14, label="real water (experiment)")
ax2.set_xlabel("O–H bond length R (Å)"); ax2.set_ylabel("H–O–H angle θ (degrees)")
ax2.set_title("The outer loop searches a plane of geometries")
ax2.legend(fontsize=8, loc="upper right"); ax2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"{len(geometries)} geometries to scan in Part A")
print("Example: R = 1.0 Å, θ = 100°  ->  PySCF geometry string:", h2o_geometry(1.0, 100))

## Cars and lanes: from H₂ to H₂O
**Recap of H₂ (Module 3, Lesson 3; Lab 4, Part C1).** Each spin orbital is a **lane** and each electron is a **car**. H₂ has two roads, a low-energy **bond** road and a high-energy **anti-bond** road, and each road has a spin-up lane and a spin-down lane: **4 lanes = 4 qubits** (`1` = car in the lane, `0` = empty). The two cars start in the bond lanes, the Hartree–Fock state $|1010\rangle$. The ground state also includes a little bit of the **double lane change**, both cars moving to the anti-bond lanes together, $|0101\rangle$.

**Water uses the same picture.** Its atoms provide **7 roads**, each with a spin-up and a spin-down lane: **14 lanes = 14 qubits**, and there are **10 cars**. Lanes higher up cost more energy, so in the Hartree–Fock state the 10 cars fill the 5 lowest roads (one spin-up and one spin-down car on each):

| Road (bottom = lowest energy) | What it does, in plain words | Cars in Hartree–Fock |
|---|---|---|
| O core | deep inside the oxygen atom ("underground garage") | 2 |
| O inner | close to the oxygen, hardly involved in bonding | 2 |
| bond road 1 | holds the O–H bonds together | 2 |
| bond road 2 | holds the O–H bonds together | 2 |
| lone-pair road | stays on the oxygen; never touches the hydrogens | 2 |
| anti-bond road 1 | empty "fast lane" that weakens the bonds | 0 |
| anti-bond road 2 | empty "fast lane" that weakens the bonds | 0 |

Why 7 roads? In the STO-3G basis (the same one as Labs 5 and 6), each hydrogen brings one 1s orbital and oxygen brings five (1s, 2s, 2pₓ, 2p_y, 2p_z). When the atoms bond, these 2 + 5 = 7 atomic orbitals mix into 7 molecular orbitals, just as H₂'s two 1s orbitals became one bonding and one antibonding orbital (Module 3, Lesson 3, "Molecular & Spin Orbitals").

> **📘 Lecture link — Module 3, Lesson 3, "Representing a Configuration Mix: CI".** CI lists *every possible occupation of lanes*, and the slide warns that "this becomes huge in large molecules." For H₂ there were 4 spin-zero configurations ($\Phi_A \dots \Phi_D$). For water, 5 spin-up cars in 7 spin-up lanes can be arranged in $\binom{7}{5} = 21$ ways, and the same for spin down: $21 \times 21 = 441$ configurations. That is why we use the CC/UCC "lane change" description below instead.

You do not need to know where these names come from. What matters: water has **two bond roads and two anti-bond roads**, a "double copy" of H₂'s one bond road and one anti-bond road, plus three roads whose cars mostly stay put.

The picture below draws the lanes for H₂ and for water.

In [ ]:
# [Cell 4]
from matplotlib.patches import FancyBboxPatch, Rectangle

def draw_car(ax, x, y, color):
    "A tiny car: a rounded body and two wheels."
    ax.add_patch(FancyBboxPatch((x - 0.17, y - 0.05), 0.34, 0.13, boxstyle="round,pad=0.02", fc=color, ec="black", lw=0.8, zorder=3))
    for dx in (-0.09, 0.09):
        ax.add_patch(Circle((x + dx, y - 0.07), 0.035, fc="black", zorder=4))

def draw_roads(ax, roads, cars, title, parked=(), arrows=(), n_rows=7):
    "roads: names bottom-to-top; cars: set of (road index, 'up'/'down'); parked: road indices to grey out."
    for k, name in enumerate(roads):
        y = k * 0.55
        for side, x in (("up", 0.0), ("down", 0.9)):
            shade = "#e6e6e6" if k in parked else "#cfe2f3"
            ax.add_patch(Rectangle((x - 0.32, y - 0.13), 0.64, 0.26, fc=shade, ec="gray", lw=0.8, zorder=1))
            if (k, side) in cars:
                draw_car(ax, x, y, "gray" if k in parked else "tab:orange")
        ax.text(1.35, y, name + ("  (parked)" if k in parked else ""), va="center", fontsize=9,
                color="gray" if k in parked else "black")
    for (k1, k2) in arrows:                                   # a double lane change: both cars move up together
        for x in (0.0, 0.9):
            ax.annotate("", xy=(x + 0.25, k2 * 0.55), xytext=(x + 0.25, k1 * 0.55),
                        arrowprops=dict(arrowstyle="->", color="tab:red", lw=2, connectionstyle="arc3,rad=-0.4"))
    ax.text(0.0, len(roads) * 0.55 - 0.1, "spin ↑", ha="center", fontsize=10)
    ax.text(0.9, len(roads) * 0.55 - 0.1, "spin ↓", ha="center", fontsize=10)
    ax.set_title(title, fontsize=11)
    ax.set_xlim(-0.5, 3.4); ax.set_ylim(-0.4, n_rows * 0.55 + 0.1); ax.axis("off")   # same scale in every panel

h2o_roads = ["O core", "O inner", "bond road 1", "bond road 2", "lone-pair road", "anti-bond road 1", "anti-bond road 2"]
h2o_cars = {(k, s) for k in range(5) for s in ("up", "down")}

fig, axes = plt.subplots(1, 3, figsize=(15, 5.2), gridspec_kw={"width_ratios": [1, 1.2, 1.2]})
draw_roads(axes[0], ["bond road", "anti-bond road"], {(0, "up"), (0, "down")},
           "H₂ (Lab 4): 4 lanes, 2 cars\n→ 4 qubits", arrows=[(0, 1)])
draw_roads(axes[1], h2o_roads, h2o_cars, "H₂O, unreduced: 14 lanes, 10 cars\n→ 14 qubits")
draw_roads(axes[2], h2o_roads, h2o_cars, "H₂O after Step 1: park the cars that stay put\n→ 8 lanes, 4 moving cars",
           parked=(0, 1, 4), arrows=[(2, 5)])
plt.tight_layout(); plt.show()

- **Left (H₂):** the red arrows show the double lane change from Lab 4: both cars move from the bond lanes to the anti-bond lanes together.
- **Middle (water, unreduced):** 10 cars fill the 5 lowest roads. This is water's Hartree–Fock state.
- **Right (water, reduced):** the cars in the gray roads stay put (next section), so only the 4 orange cars move, among 8 lanes. The red arrows show one possible double lane change. It looks like **two copies of H₂**.

## The unreduced ansatz: every possible lane change
Recall the ansatz recipe from **Module 3, Lesson 4** (CC → UCC → UCCSD), which Lab 4 (Part C2) and Lab 5 (Optional) turned into code:
1. **Start** from Hartree–Fock: all cars in the lowest lanes. On the qubits, that is just X gates on the occupied lanes.
2. **Add lane changes**, each with its own control knob $\theta_k$:
   - a **single lane change**: one car moves from a filled lane to an empty lane of the same spin,
   - a **double lane change**: two cars move together.

> **📘 Lecture link — Module 3, Lesson 4.** CC builds every mix by *"lane change from a reference"*: $|\Psi\rangle = (1 + T_B + T_C + T_D)|\Phi_A\rangle$, where $T_B, T_C$ move one electron and $T_D$ moves two together. UCC uses $e^{T - T^\dagger}$ so that lanes can change *"back and forth"* and the operation is unitary, with one variational parameter $\theta$ per lane change. UCCSD keeps the singles (S) and doubles (D).

For H₂ there were only 3 possible lane changes (two singles and one double, the $T_C$, $T_B$ and $T_D$ of the lecture). For water, with 10 cars and 14 lanes, let's count them, and see how big the circuit and the Hamiltonian become.

In [ ]:
# [Cell 5]
from pyscf import gto, scf
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper, ParityMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCC, UCCSD
from qiskit_nature.units import DistanceUnit

def q0_first(circuit):
    "Hartree-Fock bitstring with lane (qubit) 0 first, the order of our pictures."
    return Statevector(circuit).probabilities_dict().popitem()[0][::-1]

def cnot_count(circuit):
    return transpile(circuit, basis_gates=["cx", "rz", "sx", "x"], optimization_level=1).count_ops().get("cx", 0)

full = PySCFDriver(atom=h2o_geometry(R_FIXED, THETA_FIXED), basis="sto3g", unit=DistanceUnit.ANGSTROM).run()
jw = JordanWignerMapper()                                     # one qubit per lane (Lab 4 / Lab 5 optional)
H_full = jw.map(full.hamiltonian.second_q_op())
hf_full = HartreeFock(full.num_spatial_orbitals, full.num_particles, jw)
uccsd_full = UCCSD(full.num_spatial_orbitals, full.num_particles, jw, initial_state=hf_full)
n_single = sum(1 for e in uccsd_full.excitation_list if len(e[0]) == 1)
n_double = sum(1 for e in uccsd_full.excitation_list if len(e[0]) == 2)

print(f"lanes (qubits)                 : {H_full.num_qubits}      (H2: 4)")
print(f"Hartree-Fock state (q0 first)  : {q0_first(hf_full)}")
print(f"   spin-up lanes q0-q6         : {q0_first(hf_full)[:7]}   spin-down lanes q7-q13: {q0_first(hf_full)[7:]}")
print(f"single lane changes            : {n_single}")
print(f"double lane changes            : {n_double}")
print(f"control knobs in the ansatz    : {uccsd_full.num_parameters}     (H2: 3)")
print(f"CNOT gates in the ansatz       : {cnot_count(uccsd_full)}   ")
print(f"Hamiltonian terms              : {len(H_full)}    (H2 with 4 qubits: 15)")
print(f"measurement circuits per energy: {len(H_full.group_commuting(qubit_wise=True))}")

Read the Hartree–Fock bitstring with the pictures: `1111100` for the spin-up lanes (5 cars in the 5 lowest roads, the 2 anti-bond lanes empty), and the same for spin-down.

Now look at the cost in the output of Cell 5: **140 control knobs and thousands of CNOTs**, with over a thousand Hamiltonian terms. (The exact CNOT count depends on your Qiskit version: newer versions compile these circuits more cleverly. It is large either way.) Compare with the H₂ circuit of Module 3, Lesson 5: *2 Rz rotations and 4 CNOT gates*. Running VQE with this would be hopeless on today's devices, and very slow even on our simulator. We need to shrink the problem, without throwing away the part of water that changes when we bend or stretch it.

## Shrinking the problem in three easy steps
### Step 1 — Park the cars that never move (14 → 8 qubits)
Look at the gray roads in the right panel above.
- The **O core** cars sit in an "underground garage", far below every other lane. Moving them out would cost so much energy that it essentially never happens.
- The **O inner** cars are also deep and stay near the oxygen.
- The **lone-pair** cars stay on the oxygen too. Their road never touches the hydrogens, so stretching the O–H bonds or bending the molecule hardly changes it.

> **📘 Lecture link — Module 3, Lesson 2.** For H₂, the ground state is the bonding configuration plus *"a tiny percentage of destructive configuration (antibonding)."* Only cars near the top of the filled lanes can afford that small move into the empty lanes. Cars deep down essentially never move, which is what justifies parking them.

So we **park** these 6 cars: we keep them fixed in their lanes and add their energy as a constant, computed classically. That removes 3 roads = 6 lanes. What is left are the 4 cars in the two **bond roads** and the two empty **anti-bond roads**: **8 lanes, 4 cars**, two copies of H₂'s picture. (Chemists call this kept part the *active space*.)

In the code, the lone-pair road is found by its defining property, "no contact with the hydrogens", because the roads can swap order in the list when the geometry changes.

In [ ]:
# [Cell 6]
def active_roads(geometry):
    "Indices of the 4 roads we keep: the two bond roads and the two anti-bond roads."
    m = gto.M(atom=geometry, basis="sto3g", verbose=0)
    hf_calc = scf.RHF(m).run()
    h_parts = [i for i, label in enumerate(m.ao_labels()) if label.split()[1] == "H"]
    contact_with_H = (hf_calc.mo_coeff[h_parts, :] ** 2).sum(axis=0)
    lone_pair = min(range(2, 5), key=lambda i: contact_with_H[i])    # roads 0 and 1 are O core and O inner
    return [i for i in range(2, 5) if i != lone_pair] + [5, 6], hf_calc.e_tot

roads, _ = active_roads(h2o_geometry(R_FIXED, THETA_FIXED))
step1 = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=4, active_orbitals=roads).transform(full)
H_step1 = jw.map(step1.hamiltonian.second_q_op())
hf_step1 = HartreeFock(4, step1.num_particles, jw)
uccd_step1 = UCC(4, step1.num_particles, "d", jw, initial_state=hf_step1)

print(f"lanes (qubits)               : {H_step1.num_qubits}")
print(f"Hartree-Fock state (q0 first): {q0_first(hf_step1)}   "
      f"(spin up: bond1 bond2 anti1 anti2 = {q0_first(hf_step1)[:4]}, spin down = {q0_first(hf_step1)[4:]})")
print(f"double lane changes (knobs)  : {uccd_step1.num_parameters}")
print(f"CNOT gates in the ansatz     : {cnot_count(uccd_step1)}")
print(f"Hamiltonian terms            : {len(H_step1)}")

The Hartree–Fock state is now `1100 1100`: in each spin direction, the two bond lanes are full and the two anti-bond lanes are empty.

From here on we keep only the **double** lane changes, as we did for H₂. The lecture noted that for H₂ the two single excitations $T_B$ and $T_C$ *"drop out ... due to spatial symmetry constraints (g/u symmetry or Brillouin's theorem)"* (Module 3, Lesson 4, UCCSD slide), and Lab 5's optional part confirmed that their parameters stay near zero. With 2 spin-up and 2 spin-down cars, there are **18** ways for two cars to move up together:
- both spin-up cars jump to the two spin-up anti-bond lanes (1 way),
- both spin-down cars do the same (1 way),
- one spin-up car and one spin-down car each jump to an anti-bond lane (2 × 2 × 2 × 2 = 16 ways).

Compare the CNOT count printed by Cell 6 with that of Cell 5: the circuit is already several times smaller. Steps 2 and 3 remove more qubits.

### Step 2 — Use the car-count rule (8 → 6 qubits)
> **📘 Lecture link — Module 3, Lesson 4, "Reduced Qubit Mapping".** For H₂ the lecture said *"Redundancy makes it possible to use 2 qubits instead of 4 (q2 copies q0; q3 copies q1),"* and defined **tapering** as *"a mathematical technique used to permanently remove redundant qubits ... by exploiting the conservation laws (symmetries) of the system."* Steps 2 and 3 are exactly that, applied to water. The bookkeeping in Step 2 is the **parity** mapping, one of the mappings listed in Module 4, Lesson 2 (*"Jordan–Wigner (JW), Bravyi–Kitaev (BK), and parity"*).

The number of cars in each spin direction **never changes**: there are always exactly 2 spin-up cars and 2 spin-down cars. Lane changes move cars, but never create or destroy them.

We can use a different bookkeeping: Instead of "is there a car in this lane?", each qubit records **"so far, is the number of cars even or odd?"**, counting lanes from the first one up to this one. The qubit at the end of the spin-up lanes then counts *all* spin-up cars, which is always 2, an even number. That qubit always reads the same value, so it carries no information and we can drop it.

The last qubit counts *all* the cars, always 4, also even, so it is dropped too. Two qubits are gone.

### Step 3 — Use the mirror rule (6 → 5 qubits)
This is the same kind of argument as the lecture's g/u symmetry for H₂: a symmetry of the molecule forbids some lane changes and makes some qubits redundant.

Water looks the same in a mirror that swaps its two hydrogens. Each road either **looks the same** in that mirror or **flips** (its left and right halves have opposite signs):

| Road | In the mirror |
|---|---|
| bond road 1 | flips |
| bond road 2 | looks the same |
| anti-bond road 1 | looks the same |
| anti-bond road 2 | flips |

The Hartree–Fock state does not flip, and neither does water's ground state. A double lane change that would make the whole picture flip can never be part of the ground state, so it is dropped: that removes **8** of the 18 double lane changes, leaving **10**. Since the "flip or not" answer is always the same, one more qubit carries no information and is removed (this is called *tapering*).

Let's check all three steps with the actual numbers.

In [ ]:
# [Cell 7]
step2 = ParityMapper(num_particles=step1.num_particles)          # car-count rule
H_step2 = step2.map(step1.hamiltonian.second_q_op())
uccd_step2 = UCC(4, step1.num_particles, "d", step2, initial_state=HartreeFock(4, step1.num_particles, step2))

step3 = step1.get_tapered_mapper(step2)                           # + mirror rule
H_step3 = step3.map(step1.hamiltonian.second_q_op())
uccd_step3 = UCC(4, step1.num_particles, "d", step3, initial_state=HartreeFock(4, step1.num_particles, step3))

stages = [("unreduced", H_full, uccsd_full), ("Step 1: park", H_step1, uccd_step1),
          ("Step 2: car count", H_step2, uccd_step2), ("Step 3: mirror", H_step3, uccd_step3)]
print(f"{'':20}{'qubits':>7}{'knobs':>7}{'CNOTs':>8}{'terms':>7}")
rows = []
for name, H, ans in stages:
    rows.append((name, H.num_qubits, ans.num_parameters, cnot_count(ans), len(H)))
    print(f"{name:20}{rows[-1][1]:7d}{rows[-1][2]:7d}{rows[-1][3]:8d}{rows[-1][4]:7d}")
print(f"{'H2 (Lab 5, 2 qubits)':20}{2:7d}{1:7d}{4:8d}{6:7d}")

From **14 qubits, 140 knobs and thousands of CNOTs** down to **5 qubits, 10 knobs and far fewer CNOTs** (see the table printed by Cell 7), and we kept the part of water that matters for bending and stretching.

Two things to keep in mind:
- After Steps 2 and 3, a qubit no longer means "is there a car in this lane?" It stores the bookkeeping (even/odd counts and the mirror rule). That is why the Hartree–Fock circuit on 5 qubits needs only a single X gate, as you will see below.
- Step 1 is an **approximation** (we assumed the parked cars never move). Steps 2 and 3 are **exact**: they only remove information that was never really there.

All constant energies (the repulsion between the nuclei and the energy of the parked cars) are added to the identity term, the *"constant energy offset"* $c_0 I$ of Module 4, Lesson 2, so every energy below is a **total** energy in Hartree, as in Lab 6.

## The big picture: the same two loops as Lab 6, with a two-parameter outer loop
> **📘 Lecture link — Module 4, Lesson 5, "Overall VQE Process".** The inner loop (*electronic structure*) finds the lowest energy for one geometry. The outer loop (*geometry optimization*) asks: *"Could a different geometry yield an even lower energy?"* and changes it, and *"the Hamiltonian changes accordingly."* For H₂O the slide's outer loop adjusts *r* and *α*.

| | **Inner loop** — electronic structure | **Outer loop** — geometry optimization |
|---|---|---|
| What changes | the 10 control knobs $\theta_1, \dots, \theta_{10}$ (the lane changes) | the bond length $R$ **and** the bond angle $\theta$ |
| What stays fixed | the Hamiltonian (one fixed geometry) | — the Hamiltonian **changes** with $(R, \theta)$ |
| Who adjusts it | COBYLA | a scan over a list of $(R, \theta)$ pairs |
| Result | the lowest energy at that geometry | the lowest-energy shape |

Careful with the symbols: $\theta_1, \dots, \theta_{10}$ are the **circuit** knobs, while $\theta$ without a subscript is the **bond angle**. In the code, the bond angle is called `theta` and the circuit knobs are a NumPy array of length 10.

## One function for any geometry
`build_h2o(R, theta)` repeats the three steps for one geometry and returns the 5-qubit Hamiltonian, the Hartree–Fock circuit, and the 10-knob ansatz. It also returns the classical Hartree–Fock energy (all cars in their lowest lanes, no lane changes), which we use as the reference for VQE.

In [ ]:
# [Cell 8]
def build_h2o(R, theta):
    "5-qubit Hamiltonian, Hartree-Fock circuit and 10-knob ansatz at geometry (R, theta)."
    geometry = h2o_geometry(R, theta)
    roads, E_hf = active_roads(geometry)                                     # which roads to keep
    problem = PySCFDriver(atom=geometry, basis="sto3g", unit=DistanceUnit.ANGSTROM).run()
    reduced = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=4,
                                     active_orbitals=roads).transform(problem)  # Step 1: park
    mapper = reduced.get_tapered_mapper(ParityMapper(num_particles=reduced.num_particles))  # Steps 2 + 3

    H = mapper.map(reduced.hamiltonian.second_q_op())
    constant = sum(reduced.hamiltonian.constants.values())    # nuclear repulsion + parked cars
    H = (H + SparsePauliOp("I" * H.num_qubits, constant)).simplify()

    hf = HartreeFock(reduced.num_spatial_orbitals, reduced.num_particles, mapper)
    ansatz = UCC(reduced.num_spatial_orbitals, reduced.num_particles, "d", mapper, initial_state=hf)
    return {"H": H, "hf": hf, "ansatz": ansatz, "E_hf": E_hf}

h2o = build_h2o(R_FIXED, THETA_FIXED)
H_ref, hf_ref, ansatz_ref = h2o["H"], h2o["hf"], h2o["ansatz"]
print("Hartree-Fock circuit (5 qubits):")
print(hf_ref.draw(output="text"))
print(f"control knobs: {ansatz_ref.num_parameters}   CNOTs: {cnot_count(ansatz_ref)}   "
      f"Hamiltonian terms: {len(H_ref)}   measurement circuits: {len(H_ref.group_commuting(qubit_wise=True))}")

One X gate prepares all four moving cars, thanks to the even/odd bookkeeping. For H₂ we needed 3 measurement circuits (Z, X and Y); water needs **34**, because its Hamiltonian has 132 terms. Terms that use the same letter (or I) on every qubit can share one circuit.

### Measuring the energy (the same recipe as Lab 6, generalized)
The recipe is unchanged from Lab 6: rotate each qubit into the basis its terms need ($H$ for X, $S^\dagger H$ for Y), measure, turn counts into parity expectation values, multiply by the coefficients, and add everything up. The only difference is 34 circuits instead of 3.

> **📘 Lecture link — Module 4, Lessons 3–4.** The energy of a trial state is the expectation value $\langle\psi|H|\psi\rangle$ (a trial state is not an energy eigenstate, so it has no definite energy). Each Pauli term is measured by mapping bit 0 → +1 and bit 1 → −1, after rotating X or Y into Z with $HZH = X$ and $SHZHS^\dagger = Y$. For H₂ the lecture asked *"Does this mean we need to run the quantum circuit three times?"* and answered *"Yes."* For water the answer is 34 times.

As in Lab 6, the circuits are transpiled **once** with symbolic parameters and all of them are sent to the simulator in **one** call.

In [ ]:
# [Cell 9]
def measurement_circuits(ansatz, H, backend):
    "One transpiled (still symbolic) circuit per group of terms that can be measured together."
    groups = H.group_commuting(qubit_wise=True)
    circuits = []
    for group in groups:
        qc = ansatz.copy()
        for q in range(qc.num_qubits):
            needed = {label[-1 - q] for label in group.paulis.to_labels()} - {"I"}
            if needed == {"X"}:              # HZH = X
                qc.h(q)
            elif needed == {"Y"}:            # S H Z H S† = Y
                qc.sdg(q); qc.h(q)
        qc.measure_all()
        circuits.append(transpile(qc, backend, optimization_level=1, seed_transpiler=SEED))
    return groups, circuits

def energy_from_counts(groups, counts_list):
    "Coefficient x parity expectation, summed over all terms of all groups."
    energy = 0.0
    for group, counts in zip(groups, counts_list):
        for label, coeff in group.to_list():
            qubits = [q for q, ch in enumerate(reversed(label)) if ch != "I"]
            energy += coeff.real * parity_expectation(counts, qubits)
    return energy

def sampled_energy(params, groups, circuits, backend, shots=SHOTS):
    "Energy expectation value (total, Hartree) estimated from shots."
    bound = [c.assign_parameters(params) for c in circuits]
    result = backend.run(bound, shots=shots, seed_simulator=next(_seed_counter)).result()
    return energy_from_counts(groups, [result.get_counts(k) for k in range(len(circuits))])

## One Hamiltonian per geometry
As in Lab 6, *adjust the geometry, and the Hamiltonian changes accordingly*. Now "the geometry" means the pair $(R, \theta)$, and `build_h2o` builds a new Hamiltonian for each pair.

Before running VQE, it helps to see the **Hartree–Fock energy** (all cars in their lowest lanes, no lane changes) over the whole $(R, \theta)$ plane. Hartree–Fock is cheap on a classical computer, so we can compute it on a fine grid and draw it as a map of the landscape the outer loop has to search. Our scan lines are drawn on top. This takes about one minute.

In [ ]:
# [Cell 10]
# Hartree-Fock energy landscape over the (R, theta) plane (classical, cheap)
R_grid = np.linspace(0.8, 1.4, 13)
theta_grid = np.linspace(80, 140, 13)
E_hf_map = np.array([[scf.RHF(gto.M(atom=h2o_geometry(R, t), basis="sto3g", verbose=0)).run().e_tot
                      for R in R_grid] for t in theta_grid])
i, j = np.unravel_index(E_hf_map.argmin(), E_hf_map.shape)

plt.figure(figsize=(7.5, 5.5))
cs = plt.contourf(R_grid, theta_grid, E_hf_map, levels=25, cmap="viridis")
plt.colorbar(cs, label="Hartree–Fock energy (Hartree)")
plt.contour(R_grid, theta_grid, E_hf_map, levels=12, colors="white", linewidths=0.5, alpha=0.6)
plt.plot(R_scan, [THETA_FIXED] * len(R_scan), "o-", color="tab:orange", label="our R scan")
plt.plot([R_FIXED] * len(theta_scan), theta_scan, "s-", color="tab:purple", label="our θ scan")
plt.plot(R_grid[j], theta_grid[i], "w*", ms=15, mec="black", label=f"lowest HF energy on this grid: R = {R_grid[j]:.2f} Å, θ = {theta_grid[i]:.0f}°")
plt.xlabel("O–H bond length R (Å)"); plt.ylabel("H–O–H angle θ (degrees)")
plt.title("H₂O energy landscape (Hartree–Fock): a valley in two dimensions")
plt.legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
plt.show()

# Hamiltonians for the geometries we will scan with VQE
problems = {}
for g in geometries:
    problems[g] = build_h2o(*g)
print(f"Built {len(problems)} Hamiltonians, each with {len(problems[geometries[0]]['H'])} Pauli terms.")

> **📘 Lecture link — Module 3, Lesson 2.** *"Nature finds the best mix just as a ball naturally rolls downhill to the bottom of a valley."* In the lecture, the valley was over the **mix** of configurations (the inner loop). Here we see a second valley, over the **geometry** (the outer loop). VQE has to roll downhill in both.

Think of this map as a **valley** seen from above, with dark colors at the bottom. The energy rises steeply if the bonds are squeezed (left side), more gently if they are stretched (right side), and only slowly if the angle changes (up and down). Bending water is "easier" than stretching it.

The bottom of the valley is near $R \approx 1.0$ Å and $\theta \approx 100°$, a bit different from real water (0.958 Å, 104.5°). That is because we use the smallest possible description of the atoms (the STO-3G basis, as in Labs 5 and 6); a richer description would land closer to the real values.

## Part A — Noiseless VQE at every geometry (inner loop + outer loop)
For each geometry $(R, \theta)$ in our two scans (outer loop) we run COBYLA (inner loop) from the Hartree–Fock state (all circuit parameters 0), estimating each energy from 4096 shots per circuit on the **ideal** simulator. COBYLA is the classical optimizer of Module 4, Lesson 4 (*"Adjust Parameters (θ's)"*), the same one as Lab 5 (Part C) and as QAOA in Module 2. With 10 knobs, COBYLA needs more evaluations than for H₂, so we allow up to 150 and use small first steps (`rhobeg=0.1`).

**One more detail from Lab 5's Reflection.** COBYLA reports the *lowest* of its roughly 70 noisy energy estimates, and the lowest of many noisy numbers is usually a lucky one, a little too low. So after COBYLA finishes, we **re-measure** the energy at its final parameters three times with fresh shots and report the average. That gives a fair, unbiased VQE energy without needing any exact reference.

This takes about seven minutes (11 geometries, about 35 seconds each).

In [ ]:
# [Cell 11]
ideal_sim = AerSimulator()
vqe_energy, vqe_params, vqe_evals, cobyla_best = {}, {}, {}, {}

print(f"{'R (Å)':>6}{'θ (deg)':>9}{'Hartree-Fock':>14}{'COBYLA best':>14}{'VQE (re-meas.)':>15}{'VQE - HF (mHa)':>16}{'evals':>7}")
for g in geometries:                                       # outer loop: geometry (R, theta)
    H, ans = problems[g]["H"], problems[g]["ansatz"]
    groups, circuits = measurement_circuits(ans, H, ideal_sim)
    result = minimize(                                     # inner loop: 10 control knobs
        sampled_energy, x0=np.zeros(ans.num_parameters), args=(groups, circuits, ideal_sim),
        method="COBYLA", options={"maxiter": 150, "rhobeg": 0.1},
    )
    vqe_params[g], vqe_evals[g], cobyla_best[g] = result.x, result.nfev, result.fun
    # Re-measure at the final parameters with fresh shots (removes the "lucky estimate" bias)
    vqe_energy[g] = np.mean([sampled_energy(result.x, groups, circuits, ideal_sim) for _ in range(3)])
    E_hf = problems[g]["E_hf"]
    print(f"{g[0]:6.2f}{g[1]:9.0f}{E_hf:14.5f}{cobyla_best[g]:14.5f}{vqe_energy[g]:13.5f}"
          f"{1000 * (vqe_energy[g] - E_hf):16.1f}{vqe_evals[g]:7d}")

best_geom = min(vqe_energy, key=vqe_energy.get)
print(f"\nLowest VQE energy: {vqe_energy[best_geom]:.5f} Ha at R = {best_geom[0]} Å, θ = {best_geom[1]}°")

In [ ]:
# [Cell 12]
fig, (axR, axT) = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
row = [(R, THETA_FIXED) for R in R_scan]
col = [(R_FIXED, t) for t in theta_scan]

axR.plot(R_scan, [problems[g]["E_hf"] for g in row], "^:", color="gray", label="Hartree–Fock")
axR.plot(R_scan, [vqe_energy[g] for g in row], "o-", color="tab:orange", label="noiseless VQE")
axR.set_xlabel("O–H bond length R (Å)"); axR.set_ylabel("Energy (Hartree)")
axR.set_title(f"Stretching both bonds (θ = {THETA_FIXED}°)")

axT.plot(theta_scan, [problems[g]["E_hf"] for g in col], "^:", color="gray", label="Hartree–Fock")
axT.plot(theta_scan, [vqe_energy[g] for g in col], "s-", color="tab:purple", label="noiseless VQE")
axT.set_xlabel("H–O–H angle θ (degrees)")
axT.set_title(f"Opening the angle (R = {R_FIXED} Å)")

for ax, pts, x_of in ((axR, row, lambda g: g[0]), (axT, col, lambda g: g[1])):
    b = min(pts, key=vqe_energy.get)
    ax.plot(x_of(b), vqe_energy[b], "r*", ms=15, label="lowest VQE energy in this scan")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.suptitle("H₂O: the outer loop over two geometry parameters (noiseless VQE)")
plt.tight_layout(); plt.show()

**What to notice**
- **Stretching the bonds** (left) gives the familiar shape from H₂: squeeze the atoms and the energy shoots up, pull them apart and it rises toward broken bonds, with a minimum in between, here near $R = 1.0$ Å.
- **Opening the angle** (right) also has a minimum, near $\theta = 100°$, but the curve is **flatter**: bending costs less energy than stretching. Notice the same y-axis on both panels.
- **VQE lies below Hartree–Fock everywhere.** The energy of any trial state can never be lower than the true ground-state energy (the variational principle behind VQE), so a lower energy means a better description: VQE's lane changes (cars moving a little into the anti-bond lanes) capture something that Hartree–Fock, with every car frozen in its lowest lane, leaves out.
- **The gap tends to grow as the bonds stretch**, from roughly 15 mHa at 0.8 Å to around 100 mHa at 1.4 Å (single points can wobble, see the next bullet). At long $R$ the bond lanes and the anti-bond lanes come closer in energy, so lane changes become cheap and happen more, and the "everyone in the lowest lane" picture of Hartree–Fock gets worse. That is the same trend we saw for H₂ in Lab 6.
- **COBYLA's best value is always a bit lower than the re-measured energy.** That is the lucky-estimate effect described above. Even the re-measured energies carry a few mHa of shot noise, and COBYLA does not always tune all 10 knobs equally well, so neighboring points near the bottom of the valley (for example 100° and 110°) can swap order. The overall shape is clear, though.

## Part B — The same circuits on a noisy (simulated) IBM device
> **📘 Lecture link — Module 3, Lesson 1, "Hybrid Algorithms for Today's Hardware (NISQ)".** VQE was designed for today's noisy, intermediate-scale quantum (NISQ) devices: the quantum computer scores each candidate and the classical computer decides the next direction. This part shows how much noise such a device can add.

In Lab 6 we ran the whole VQE loop on the FakeSherbrooke noise model. For water that would take far too long (each noisy energy needs 34 circuits with hundreds of gates). Instead we take the parameters VQE found in Part A and measure their energy under FakeSherbrooke's noise, at every geometry.

We also measure the plain **Hartree–Fock circuit**, which has no two-qubit gates at all, to see how much error comes from measurement alone.

First, how big is our circuit once it is translated for the device?

In [ ]:
# [Cell 13]
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

fake_sherbrooke = FakeSherbrooke()
sherbrooke_simulator = AerSimulator.from_backend(fake_sherbrooke)   # simulator with Sherbrooke's noise

_, sample = measurement_circuits(ansatz_ref, H_ref, sherbrooke_simulator)
ops = sample[0].count_ops()
print(f"H2O ansatz on the device: {ops.get('ecr', 0)} two-qubit (ECR) gates, depth {sample[0].depth()}")
print("H2 ansatz on the device : 4 two-qubit (ECR) gates (Lab 6)")

Hundreds of two-qubit gates, compared with 4 for H₂. Every one of them has a small chance of error, and the errors add up. Let's see what that does to the energy. This takes about two minutes.

In [ ]:
# [Cell 14]
noisy_hf, noisy_vqe, scrambled = {}, {}, {}
print(f"{'R (Å)':>6}{'θ (deg)':>9}{'noiseless VQE':>15}{'noisy HF':>11}{'noisy VQE':>11}")
for g in geometries:
    H, hf, ans = problems[g]["H"], problems[g]["hf"], problems[g]["ansatz"]
    g_hf, c_hf = measurement_circuits(hf, H, sherbrooke_simulator)
    g_an, c_an = measurement_circuits(ans, H, sherbrooke_simulator)
    noisy_hf[g] = sampled_energy([], g_hf, c_hf, sherbrooke_simulator)
    noisy_vqe[g] = sampled_energy(vqe_params[g], g_an, c_an, sherbrooke_simulator)
    # Energy of a completely scrambled state = the identity coefficient (every other term averages to 0)
    scrambled[g] = H.coeffs[H.paulis.to_labels().index("I" * H.num_qubits)].real
    print(f"{g[0]:6.2f}{g[1]:9.0f}{vqe_energy[g]:15.4f}{noisy_hf[g]:11.4f}{noisy_vqe[g]:11.4f}")

In [ ]:
# [Cell 15]
fig, (axR, axT) = plt.subplots(1, 2, figsize=(12, 4.8), sharey=True)
for ax, pts, xs, xlabel, title in (
        (axR, row, R_scan, "O–H bond length R (Å)", f"Stretching both bonds (θ = {THETA_FIXED}°)"),
        (axT, col, theta_scan, "H–O–H angle θ (degrees)", f"Opening the angle (R = {R_FIXED} Å)")):
    ax.plot(xs, [vqe_energy[g] for g in pts], "o-", label="noiseless VQE (Part A)")
    ax.plot(xs, [noisy_hf[g] for g in pts], "v-", label="Hartree–Fock circuit, FakeSherbrooke")
    ax.plot(xs, [noisy_vqe[g] for g in pts], "s-", label="VQE circuit, FakeSherbrooke")
    ax.plot(xs, [scrambled[g] for g in pts], "--", color="gray", label="completely scrambled state")
    ax.set_xlabel(xlabel); ax.set_title(title); ax.grid(alpha=0.3)
axR.set_ylabel("Energy (Hartree)"); axR.legend(fontsize=8)
plt.suptitle("H₂O: what noise does to the energy landscape"); plt.tight_layout(); plt.show()

g = best_geom
print(f"At R = {g[0]} Å, θ = {g[1]}°:")
print(f"  energy VQE gained over Hartree-Fock (noiseless): {1000 * (problems[g]['E_hf'] - vqe_energy[g]):8.1f} mHa")
print(f"  noise error on the Hartree-Fock circuit         : {1000 * (noisy_hf[g] - problems[g]['E_hf']):8.1f} mHa")
print(f"  noise error on the VQE circuit                  : {1000 * (noisy_vqe[g] - vqe_energy[g]):8.1f} mHa")

**What to notice**
- **Even the Hartree–Fock circuit is pulled up**, although it has no two-qubit gates. That error comes purely from **measurement (readout) errors**: sometimes a 0 is read as 1, or the reverse, and with 34 circuits those mistakes add up.
- **The VQE circuit is far worse.** Its hundreds of two-qubit gates scramble the state so much that its energy lands near the gray "completely scrambled" line. A scrambled state has no memory of the chemistry, so its energy is just the constant part of the Hamiltonian.
- **The geometry information is damaged too.** The noisy Hartree–Fock curve is shifted up by roughly the same amount everywhere, so it roughly keeps its shape, with a minimum near $R$ = 1.0–1.1 Å. The noisy VQE curve does not: along $R$ it just keeps falling, following the scrambled line, and along $\theta$ it is flat. The noisy VQE would find *no* equilibrium geometry at all. Compare the sizes: VQE gained a few tens of mHa over Hartree–Fock, while noise adds well over a thousand.
- This is why real hardware experiments on molecules like water use much shorter circuits (for example, "hardware-efficient" ansätze or adaptive methods that keep only the most important excitations) together with **error mitigation**.

## Optional — Run on a real IBM quantum computer
Everything so far used simulators. If you have an IBM Quantum account, you can measure the water energy on a real device.

**What we run.** Running the whole VQE loop over all geometries on hardware would need many thousands of circuits, far beyond the free plan. So, as in Part B, we take the parameters VQE found in Part A and measure only **two** states at the best geometry from Part A:
1. the **Hartree–Fock** circuit (no two-qubit gates), and
2. the **VQE** circuit with the optimized parameters (hundreds of two-qubit gates).

Each needs the 34 measurement circuits, so this is 2 × 34 = 68 circuits with 2048 shots each, roughly **1–2 minutes of QPU time**, plus however long you wait in the queue.

**Before you start**
- The free **Open plan** gives 10 minutes of QPU time per 28 days and does not allow sessions, so we submit both jobs in **batch mode**, exactly as in the QAOA lab.
- Create an API key on [quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) and copy your instance CRN.
- **Never paste a real API key into a notebook you share or submit.** Use the placeholders below only in your private copy, or save your account once with `QiskitRuntimeService.save_account(...)` and leave `token` and `instance` out.
- Change `RUN_ON_HARDWARE` to `True` to run the cell. Otherwise it is skipped.

Today's IBM devices may use a different native two-qubit gate than FakeSherbrooke (CZ instead of ECR), so the code counts whichever one the device uses.

In [ ]:
# [Cell 16]
RUN_ON_HARDWARE = False      # change to True after filling in your credentials
HW_SHOTS = 2048

if RUN_ON_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler, Batch

    service = QiskitRuntimeService(channel="ibm_quantum_platform",
                                   token="<YOUR_API_KEY>", instance="<YOUR_INSTANCE_CRN>")
    backend = service.least_busy(operational=True, simulator=False, min_num_qubits=5)
    print("Running on:", backend.name)

    g = best_geom
    H, hf, ans = problems[g]["H"], problems[g]["hf"], problems[g]["ansatz"]
    g_hf, c_hf = measurement_circuits(hf, H, backend)          # transpiled for this device
    g_an, c_an = measurement_circuits(ans, H, backend)
    vqe_circuits = [c.assign_parameters(vqe_params[g]) for c in c_an]
    two_qubit = sum(n for gate, n in vqe_circuits[0].count_ops().items() if gate in ("ecr", "cz", "cx"))
    print(f"VQE circuit on {backend.name}: {two_qubit} two-qubit gates, depth {vqe_circuits[0].depth()}")

    with Batch(backend=backend) as batch:                     # batch mode: allowed on the Open plan
        sampler = Sampler(mode=batch)
        job_hf = sampler.run(c_hf, shots=HW_SHOTS)
        job_vqe = sampler.run(vqe_circuits, shots=HW_SHOTS)
        print("Job IDs:", job_hf.job_id(), job_vqe.job_id())
        print("Waiting for the device (queue time varies)...")
        result_hf, result_vqe = job_hf.result(), job_vqe.result()

    hw_hf = energy_from_counts(g_hf, [r.data.meas.get_counts() for r in result_hf])
    hw_vqe = energy_from_counts(g_an, [r.data.meas.get_counts() for r in result_vqe])

    print(f"\nR = {g[0]} Å, θ = {g[1]}°, total energy (Ha)  {'noiseless':>10}{'FakeSherbrooke':>16}{backend.name:>16}")
    print(f"Hartree-Fock circuit                  {problems[g]['E_hf']:10.4f}{noisy_hf[g]:16.4f}{hw_hf:16.4f}")
    print(f"VQE circuit                           {vqe_energy[g]:10.4f}{noisy_vqe[g]:16.4f}{hw_vqe:16.4f}")

    labels = ["noiseless", "FakeSherbrooke", backend.name]
    x = np.arange(3)
    plt.figure(figsize=(8, 4.5))
    plt.plot(x, [problems[g]["E_hf"], noisy_hf[g], hw_hf], "v", ms=11, label="Hartree–Fock circuit")
    plt.plot(x, [vqe_energy[g], noisy_vqe[g], hw_vqe], "o", ms=11, label="VQE circuit")
    plt.axhline(scrambled[g], color="gray", ls="--", label="completely scrambled state")
    plt.xticks(x, labels); plt.xlim(-0.5, 2.5); plt.ylabel("Energy (Hartree)")
    plt.title(f"H₂O at R = {g[0]} Å, θ = {g[1]}°: simulator vs. real hardware")
    plt.legend(fontsize=8); plt.grid(alpha=0.3); plt.show()
else:
    print("Skipped. Set RUN_ON_HARDWARE = True (and fill in your credentials) to run on IBM hardware.")

**What to look for (if you ran it)**
- Does the real device behave like FakeSherbrooke? Remember that a fake backend is only a **snapshot** of one device's error rates; today's devices, especially the newer Heron processors, often have lower errors.
- How far is the hardware **Hartree–Fock** energy above the noiseless Hartree–Fock value? That gap is mostly readout error.
- Is the hardware **VQE** energy near the gray "scrambled" line, or does some of the chemistry survive? Compare the noise error with the few tens of mHa that VQE gained over Hartree–Fock.
- If your job sits in the queue for a long time, you can close the notebook and get the results later with `service.job("<JOB_ID>").result()`.

## Reflection
Explain the difference between:
- the inner VQE loop at a fixed geometry, and
- the outer geometry loop over $(R, \theta)$,

and explain what changed when the outer loop went from one parameter (H₂) to two (H₂O).

The inner loop changes the circuit knobs, the lane changes (10 for water, 1 for H₂), to find the lowest electronic energy for one fixed geometry. The outer loop changes the geometry itself and repeats the electronic calculation. With two geometry parameters, the outer loop searches a **plane** instead of a line: a full grid with $n$ values of each parameter needs $n^2$ VQE runs instead of $n$. That is why we scanned one parameter at a time (11 runs) instead of a full 6 × 6 grid (36 runs).

**Two more things to think about** (from the lecture)
1. *Did we really find the minimum?* Scanning one parameter at a time works well when the valley is aligned with the axes, as it nearly is for water. In general, a geometry optimizer moves **all** parameters at once, often using the forces on the atoms (energy gradients). And if we dropped our assumption $R_1 = R_2$, the outer loop would have three parameters; for larger molecules, it may adjust many atomic coordinates at the same time.
2. *VQE is expensive* (Module 4, Lesson 5). The lecture's example: 50 Pauli terms × 1,000 shots × 200 evaluations = 10 million shots for one geometry, and × 100 outer-loop steps = 1 billion. For us, for one geometry, COBYLA used tens of evaluations × 34 circuits × 4096 shots, about 10 million shots, and we did 11 geometries. That is over 100 million shots for a molecule with only 4 moving cars. Yet water is tiny: real chemistry problems have far more cars and lanes, which is exactly where VQE's ability to avoid exact diagonalization is supposed to pay off.

## Review questions
Answer in a few sentences each. Use the outputs of your own run (numbers can differ slightly from a classmate's because of shot noise and Qiskit versions).

**Q1. Cars, lanes and lane changes** (Cell 5; Module 3, Lessons 3–4)
Cell 5 prints the Hartree–Fock bitstring of the unreduced problem and the numbers of single and double lane changes.
(a) Which lanes hold cars in the Hartree–Fock state, and why is the pattern the same for spin up and spin down?
(b) Using the CC idea of *"lane change from a reference"*, explain why there are exactly **20** single lane changes. (Hint: how many filled and empty lanes does each spin direction have?)
(c) In the lecture, CI listed every possible occupation of lanes. Why is CC/UCC a more practical description for water than CI?

**Q2. Three steps to 5 qubits** (Cell 6 and Cell 7; Module 3, Lesson 4, "Reduced Qubit Mapping")
(a) Which of the three steps is an **approximation**, and which are **exact**? Explain in terms of cars and lanes.
(b) For H₂, the lecture said "q2 copies q0; q3 copies q1," so 4 qubits became 2. Which step (or steps) in water's reduction uses the same kind of redundancy?
(c) Using the table printed by Cell 7, does every step reduce the number of CNOTs? Which step makes the biggest difference to the circuit size, and why?

**Q3. Measuring the energy** (Cell 8 and Cell 9; Module 4, Lessons 3–4)
For H₂ the lecture asked whether we need to run the circuit three times and answered yes.
(a) How many measurement circuits does water need per energy evaluation (Cell 8), and why more than three?
(b) Look at `measurement_circuits` in Cell 9. What gates are added before measurement for a qubit that needs X, and for one that needs Y? Which identities from the lecture justify them?
(c) Estimate the number of shots for **one** energy evaluation of water at 4096 shots per circuit.

**Q4. Inner loop and outer loop** (Cell 11; Module 4, Lesson 5)
(a) In Cell 11, which line is the **outer loop** and which call runs the **inner loop**? What changes in each, and what stays fixed?
(b) We scanned one geometry parameter at a time (11 geometries). How many VQE runs would a full 6 × 6 grid need? How many parameters would the outer loop have if we dropped the assumption that both O–H bonds are equal, as the lecture slide mentions?
(c) Using the Part A table, at which geometry is the VQE energy lowest? How does it compare with real water (0.958 Å, 104.5°), and why might they differ?

**Q5. Noise** (Cell 14 and Cell 15; Module 3, Lesson 1 and Lab 6, Part B)
(a) The Hartree–Fock circuit has no two-qubit gates, yet its noisy energy is far too high (Cell 15 output). What kind of error causes this?
(b) Why does the noisy VQE energy land close to the gray "completely scrambled" line, and why does that line equal the identity coefficient of the Hamiltonian?
(c) In Lab 6, the noisy H₂ curve still showed roughly the right bond length. Does the noisy H₂O VQE curve still show a minimum? Explain the difference using the number of two-qubit gates printed by Cell 13.